# Trabajo Práctico 2 - Grupo 02

### Modelo XLM-RoBERTa-Base — LLM Augmented + lr=3e-5 (Kaggle)

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen

**Modelo:** `xlm-roberta-base` (125M parámetros, multilingüe, Meta AI)

**Dataset:** train aumentado con reseñas sintéticas generadas por LLM (balanceado)

**Hiperparámetros:** `lr=3e-5` | `epochs=5` | `max_length=128` | `batch_size=32`

**Motivación:** combina los dos hallazgos más importantes:
- `xlm-roberta-base + original` fue el mejor modelo hasta ahora (0.72765)
- `lr=3e-5 + ep=5` mejoró BETO de 0.72048 → 0.72596 (+0.005)
- El dataset LLM augmented mejoró BETO de 0.72048 → 0.72463 (+0.004)

La hipótesis es que combinar los tres factores en xlm-roberta-base debería superar 0.72765.

## 1. Setup

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
print("CUDA disponible:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Imports y funciones auxiliares

In [ ]:
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix

SEED = 42
VAL_SIZE = 0.2
CLASS_NAMES = ["negativa", "neutra", "positiva"]

np.random.seed(SEED)
torch.manual_seed(SEED)


def clean_minimal(text: str) -> str:
    text = unicodedata.normalize("NFKC", str(text))
    return re.sub(r"\s+", " ", text).strip()


def get_split(df, text_col="text", label_col="label"):
    return train_test_split(
        df[text_col].to_numpy(), df[label_col].to_numpy(),
        test_size=VAL_SIZE, random_state=SEED,
        stratify=df[label_col].to_numpy(),
    )


def evaluate(name, y_true, y_pred, hyperparams=None):
    f1_macro = f1_score(y_true, y_pred, average="macro")
    print(f"=== {name} ===")
    if hyperparams:
        print(f"Hiperparametros: {hyperparams}")
    print(f"\nF1-macro: {f1_macro:.4f}\n")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))
    cm = confusion_matrix(y_true, y_pred)
    print("Matriz de confusion:")
    print(pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES))
    return f1_macro

print("Funciones cargadas.")

## 3. Configuración

In [ ]:
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME  = "xlm-roberta-base"
MAX_LENGTH  = 128
BATCH_SIZE  = 32
NUM_EPOCHS  = 3
LR          = 2e-5

print(f"Device:  {DEVICE}")
print(f"Modelo:  {MODEL_NAME}")
print(f"Config:  lr={LR} | epochs={NUM_EPOCHS} | max_length={MAX_LENGTH} | batch={BATCH_SIZE}")

## 4. Carga de datos

**Ajustar `DATA_DIR`** al path del dataset subido a Kaggle.
Click derecho en el archivo en el panel "Input" → "Copy file path".

In [ ]:
DATA_DIR = "/kaggle/input/datasets/mateogonzalezpautaso/tp2-dataset"

train_df = pd.read_csv(f"{DATA_DIR}/train_augmented_llm_balanced.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")

print(f"Train: {len(train_df):,} filas")
print(f"Distribucion:\n{train_df['label'].value_counts().sort_index()}")
print(f"\nProporciones:\n{train_df['label'].value_counts(normalize=True).sort_index().round(3)}")

X_train_raw, X_val_raw, y_train, y_val = get_split(train_df)

print("\nAplicando clean_minimal...")
X_train = np.array([clean_minimal(t) for t in X_train_raw])
X_val   = np.array([clean_minimal(t) for t in X_val_raw])
X_test  = np.array([clean_minimal(t) for t in test_df["text"].values])
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

## 5. Tokenización

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Ejemplo de tokenizacion:")
for t in ["No funciona bien, muy decepcionante.", "Excelente producto."]:
    print(f"  '{t}' -> {tokenizer.tokenize(t)}")

class ResenasDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_length=128):
        self.encodings = tokenizer(
            list(texts), truncation=True,
            padding="max_length", max_length=max_length,
            return_tensors="pt",
        )
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

print("\nTokenizando...")
train_dataset = ResenasDataset(X_train, y_train, tokenizer, MAX_LENGTH)
val_dataset   = ResenasDataset(X_val,   y_val,   tokenizer, MAX_LENGTH)
test_dataset  = ResenasDataset(X_test,  None,    tokenizer, MAX_LENGTH)
print("Listo.")

## 6. Modelo y métricas

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True,
)
model = model.to(DEVICE)

total = sum(p.numel() for p in model.parameters())
print(f"Parametros totales: {total:,}")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    f1_per = f1_score(labels, preds, average=None, zero_division=0)
    return {
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_neg": float(f1_per[0]),
        "f1_neu": float(f1_per[1]),
        "f1_pos": float(f1_per[2]),
    }

## 7. Fine-tuning

In [ ]:
OUTPUT_DIR = "/kaggle/working/red_neuronal_xlm_roberta_base_llm_lr2e5_ep3_v10"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    dataloader_num_workers=2,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_only_model=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    save_total_limit=1,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"Iniciando fine-tuning...")
print(f"  Modelo:   {MODEL_NAME}")
print(f"  Dataset:  LLM aumentado ({len(X_train):,} train)")
print(f"  LR:       {LR}")
print(f"  Epochs:   {NUM_EPOCHS} (early stopping patience=2)")
trainer.train()

## 8. Evaluación

In [ ]:
preds_output = trainer.predict(val_dataset)
y_pred = np.argmax(preds_output.predictions, axis=1)

evaluate("red_neuronal_xlm_roberta_base_llm_lr2e5_ep3_v10", y_val, y_pred,
         hyperparams={
             "model": MODEL_NAME, "epochs": NUM_EPOCHS,
             "lr": LR, "max_length": MAX_LENGTH,
             "batch_size": BATCH_SIZE, "dataset": "llm_augmented",
         })

In [ ]:
history  = trainer.state.log_history
eval_f1  = [(x["epoch"], x["eval_f1_macro"]) for x in history if "eval_f1_macro" in x]
eval_neu = [(x["epoch"], x["eval_f1_neu"])   for x in history if "eval_f1_neu"   in x]

fig, ax = plt.subplots(figsize=(9, 4))
if eval_f1:
    epochs, f1s = zip(*eval_f1)
    ax.plot(epochs, f1s, marker="o", label="F1-macro", color="#1f77b4", linewidth=2)
    for e, f in zip(epochs, f1s):
        ax.annotate(f"{f:.4f}", (e, f), textcoords="offset points", xytext=(0, 8))
if eval_neu:
    epochs, f1s = zip(*eval_neu)
    ax.plot(epochs, f1s, marker="s", label="F1-neutra", color="#7f7f7f", linestyle="--")

ax.axhline(y=0.72765, color="red", linestyle=":", alpha=0.7,
           label="Mejor actual (xlm-roberta-base + original): 0.72765")
ax.set_xlabel("Epoch"); ax.set_ylabel("F1")
ax.set_title("XLM-RoBERTa-Base + LLM aug + lr=2e-5 — F1 por epoca")
ax.legend(); plt.tight_layout(); plt.show()

## 9. Guardado y submission

In [ ]:
# Guardar modelo
SAVE_DIR = "/kaggle/working/red_neuronal_xlm_roberta_base_llm_lr2e5_ep3_final"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Modelo guardado en {SAVE_DIR}")

In [ ]:
# Submission
preds_test  = trainer.predict(test_dataset)
y_test_pred = np.argmax(preds_test.predictions, axis=1)

sub = pd.DataFrame({"id": test_df["id"].values, "label": y_test_pred.astype(int)})
sub.to_csv("/kaggle/working/submission_red_neuronal_xlm_roberta_base_llm_lr2e5_ep3_v10.csv", index=False)

dist = sub["label"].value_counts(normalize=True).sort_index()
print(f"Guardado: submission_xlm_roberta_base_llm_lr3e5_ep5.csv ({len(sub)} predicciones)")
print(f"Distribucion: {', '.join(f'clase {k}: {v:.1%}' for k, v in dist.items())}")